# 05 — Final evaluation

Notebook này tổng hợp chất lượng dự báo và kết quả mô phỏng tồn kho của baseline và LSTM trên cùng holdout period. Mục đích là trả lời hai câu hỏi riêng:
1. LSTM dự báo doanh số tốt hơn baseline không?
2. Chính sách tồn kho sử dụng dự báo nào cho tổng chi phí thấp hơn theo các giả định đã chọn?

Chạy notebook `02_baseline.ipynb`, `03_lstm.ipynb`, `04_inventory_optimization.ipynb` trước notebook này.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = ARTIFACT_DIR / "results"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
for folder in (RESULT_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

LOOKBACK = 30
STORE_ID = 1

from src.data import load_rossmann_data, chronological_split
from src.features import (
    create_calendar_features, prepare_features, get_feature_columns,
    fit_feature_scaler, transform_features, fit_target_scaler,
    transform_target, inverse_transform_target, make_sequences,
    DEFAULT_LOOKBACK
)

train_df, official_test_df = load_rossmann_data(DATA_DIR, store_id=STORE_ID)
train_df = train_df.sort_values("Date").reset_index(drop=True)
splits = chronological_split(train_df, train_ratio=0.70, val_ratio=0.15, holdout_ratio=0.15)
print(f"Store: {STORE_ID} | labeled rows: {len(train_df):,}")
print(f"Train:      {splits['train']['Date'].min().date()} -> {splits['train']['Date'].max().date()} ({len(splits['train'])})")
print(f"Validation: {splits['validation']['Date'].min().date()} -> {splits['validation']['Date'].max().date()} ({len(splits['validation'])})")
print(f"Holdout:    {splits['holdout']['Date'].min().date()} -> {splits['holdout']['Date'].max().date()} ({len(splits['holdout'])})")

## 1. So sánh forecast metrics trên cùng ngày

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "WAPE_pct": np.nan}
    mae = np.mean(np.abs(y_true-y_pred))
    rmse = np.sqrt(np.mean((y_true-y_pred)**2))
    denom = np.sum(np.abs(y_true))
    wape = 100*np.sum(np.abs(y_true-y_pred))/denom if denom else np.nan
    return {"MAE": mae, "RMSE": rmse, "WAPE_pct": wape}

baseline_path = RESULT_DIR / "baseline_holdout_predictions.csv"
lstm_path = RESULT_DIR / "lstm_holdout_predictions.csv"
if not baseline_path.exists() or not lstm_path.exists():
    raise FileNotFoundError("Run notebooks 02 and 03 first.")

base = pd.read_csv(baseline_path, parse_dates=["Date"])
lstm = pd.read_csv(lstm_path, parse_dates=["Date"])
joined = base[["Date", "Sales", "SeasonalNaive_7d", "MovingAverage_30d"]].merge(
    lstm[["Date", "LSTM_Prediction"]], on=["Date", "Sales"], how="inner"
).sort_values("Date")

metric_rows = []
for name, col in [
    ("Seasonal Naive (7d)", "SeasonalNaive_7d"),
    ("Moving Average (30d)", "MovingAverage_30d"),
    ("LSTM", "LSTM_Prediction"),
]:
    metric_rows.append({"model": name, **regression_metrics(joined["Sales"], joined[col])})
forecast_comparison = pd.DataFrame(metric_rows).sort_values("WAPE_pct")
display(forecast_comparison)

## 2. So sánh biểu đồ dự báo

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(joined["Date"], joined["Sales"], label="Actual", linewidth=2)
plt.plot(joined["Date"], joined["SeasonalNaive_7d"], label="Seasonal Naive (7d)", alpha=.8)
plt.plot(joined["Date"], joined["MovingAverage_30d"], label="Moving Average (30d)", alpha=.8)
plt.plot(joined["Date"], joined["LSTM_Prediction"], label="LSTM", alpha=.8)
plt.title("Rossmann Store 1 — Holdout comparison")
plt.xlabel("Date"); plt.ylabel("Sales"); plt.legend(ncol=2); plt.tight_layout(); plt.show()

## 3. Đánh giá inventory metrics

In [ ]:
inventory_path = RESULT_DIR / "inventory_policy_comparison.csv"
if inventory_path.exists():
    inventory_comparison = pd.read_csv(inventory_path)
    display(inventory_comparison.sort_values("total_cost"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(inventory_comparison["forecast_policy"], inventory_comparison["total_cost"])
    axes[0].set_title("Total simulated inventory cost")
    axes[0].set_ylabel("Cost units")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].bar(inventory_comparison["forecast_policy"], inventory_comparison["fill_rate_pct"])
    axes[1].set_title("Fill rate")
    axes[1].set_ylabel("Percent")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout(); plt.show()
else:
    print("Inventory results not found. Run 04_inventory_optimization.ipynb.")

## 4. Xuất bảng tổng hợp và ghi nhận kết luận

In [ ]:
forecast_comparison.to_csv(RESULT_DIR / "final_forecast_comparison.csv", index=False)

print("Forecast comparison:")
display(forecast_comparison)
if inventory_path.exists():
    print("Inventory comparison:")
    display(inventory_comparison.sort_values("total_cost"))

best_forecast = forecast_comparison.iloc[0]["model"]
print(f"Lowest WAPE on holdout: {best_forecast}")
print("Do not conclude that LSTM is best unless its holdout metrics support that claim.")
print("Inventory costs are conditional on the assumptions configured in notebook 04.")

## Checklist trước khi đưa kết quả vào báo cáo
- [ ] Ghi rõ Store 1, giai đoạn dữ liệu và cách chia theo thời gian.
- [ ] Báo cáo baseline và LSTM trên cùng các ngày holdout.
- [ ] Chỉ chọn hyperparameters bằng train/validation, không chọn bằng holdout.
- [ ] Công bố các giả định về lead time, order quantity và chi phí.
- [ ] Phân biệt kết quả dự báo (MAE/RMSE/WAPE) với kết quả vận hành (cost/fill rate/stockout).
- [ ] Nêu rõ hạn chế: một cửa hàng, dữ liệu lịch sử, giả định chi phí và mô phỏng lost-sales.